# Een Herbruikbare Actuariële Tariferingsbibliotheek Bouwen met PROC FCMP

## Samenvatting

Een schadeverzekeraar (property-and-casualty) legt zijn kernrekenwerk voor tarifering — zuivere premie, kosten-/risico-opslag, credibiliteitsmenging volgens beperkte fluctuatie, en verdisconteerde reserveraming — vast als aangepaste functies en een subroutine met meerdere uitvoerwaarden in **PROC FCMP**. De gecompileerde bibliotheek wordt geregistreerd via de systeemoptie `CMPLIB=` en vervolgens rij-voor-rij aangeroepen vanuit een DATA-stap die een synthetische portefeuille van 100 polissen tarifeert. Elk getarifeerd cijfer in dit notebook — de credibiliteitsfactor `z`, de gewogen zuivere premie, de gefactureerde premie en de contant gemaakte schadereserve — wordt geproduceerd door deze gecompileerde routines, niet door inline rekenwerk. Het resultaat: het impliciete schadepercentage komt uit op 60,5% (landelijk), 55,8% (voorstedelijk) en 63,6% (stedelijk) — in elk segment ruim onder de 100%, wat bevestigt dat de opgeslagen premie de verwachte schade dekt terwijl de tariferingsstap overzichtelijk en controleerbaar blijft.

## Gegevensbronnen

| Dataset | Rijen | Beschrijving | Kernvariabelen |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Synthetische, lopende P&C-polisportefeuille inline gegenereerd met `rand()` | `policy_id`, `region` (stedelijk/voorstedelijk/landelijk), `years_insured`, `exposure` (voertuigjaren), `claim_count` (Poisson), `incurred_loss` (gamma-ernst x aantal), `class_pure_prem` (handmatig tarief per regio) |

De DATA-stap doorloopt een breder bereik van `policy_id`, maar deze omgeving draait ongelicentieerd, dus de uitvoer is beperkt tot de eerste **100 waarnemingen** — het getarifeerde bestand hieronder weerspiegelt die 100 polissen.

# Een Herbruikbare Actuariële Tariferingsbibliotheek Bouwen met PROC FCMP

Actuarissen herhalen dezelfde berekeningen in tariferings-, reserverings- en rapportagewerk: schades omzetten naar een *zuivere premie*, *kosten- en risico-opslagen* toepassen om tot een gefactureerde premie te komen, de eigen ervaring van een individuele polis mengen met een klassetarief via *credibiliteit*, en toekomstige kasstromen *verdisconteren* naar contante waarde. Deze formules in elke DATA-stap opnieuw intypen nodigt uit tot copy-paste-fouten en maakt het wijzigen van aannames lastig.

**PROC FCMP** (de SAS-functiecompiler) laat ons elke formule eenmalig definiëren als een benoemde functie of subroutine, de gecompileerde routines opslaan in een bibliotheek, en ze vervolgens aanroepen als elke ingebouwde SAS-functie. In dit notebook doen we het volgende:

1. Compileer een kleine actuariële functiebibliotheek met `PROC FCMP`.
2. Registreer deze voor de sessie met de systeemoptie `CMPLIB=`.
3. Genereer een synthetische schadeverzekeringsportefeuille.
4. Tarifeer elke polis door onze aangepaste functies en een subroutine met meerdere uitvoerwaarden aan te roepen vanuit één enkele DATA-stap.
5. Vat de getarifeerde portefeuille samen en interpreteer deze.

## Stap 1 — Genereer een synthetische polisportefeuille

We simuleren een bestand van lopende autopolissen (de eerste 100 worden hieronder getarifeerd onder de limiet van de ongelicentieerde modus). Elke polis behoort tot een tariferingsregio met een eigen handmatige *zuivere premie* (verwachte schade per voertuigjaar). Aantallen claims volgen een Poisson-proces geschaald naar blootstelling, en ernstverdelingen volgen een gammaverdeling, zodat `incurred_loss` een realistische samengestelde (Poisson x gamma) schade is. `years_insured` stuurt later de credibiliteitsweging. Een vaste seed via `call streaminit` maakt de gegevens reproduceerbaar.

In [1]:
GEGEVENS portfolio;
    CALL streaminit(20260531);
    LENGTE region $16;
    label
        policy_id       = "Polisnummer"
        region          = "Regio"
        years_insured   = "Jaren Verzekerd"
        exposure        = "Blootstelling (voertuigjaren)"
        claim_count     = "Aantal Claims"
        incurred_loss   = "Geleden Schade"
        class_pure_prem = "Klasse Zuivere Premie";
    DOE policy_id = 10001 TOT 12000;
        /* Wijs een tariferingsregio toe */
        u = rand('uniform');
        ALS u < 0.40 DAN DOE; region = 'STEDELIJK';    base_pp = 820; lambda = 0.18; EINDE;
        ANDERS ALS u < 0.75 DAN DOE; region = 'VOORSTEDELIJK'; base_pp = 540; lambda = 0.11; EINDE;
        ANDERS DOE; region = 'LANDELIJK';    base_pp = 360; lambda = 0.07; EINDE;

        /* Looptijd (jaren verzekerd) en blootstelling (voertuigjaren) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Samengesteld claimproces: Poisson-frequentie, gamma-ernst */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        DOE c = 1 TOT claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        EINDE;
        incurred_loss = round(incurred_loss, 0.01);

        /* Handmatige klasse zuivere premie voor de blootstelling van deze polis */
        class_pure_prem = round(base_pp * exposure, 0.01);

        UITVOER;
    EINDE;
    BEWAREN policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=portfolio(obs=8) noobs;
    TITEL "Eerste 8 Gesimuleerde Polissen";
UITVOEREN;


                                             Eerste 8 Gesimuleerde Polissen                                             

policy_id         region  years_insured  exposure  claim_count  incurred_loss  class_pure_prem
    10001  LANDELIJK                  5      1.36            0              0            489.6
    10002  STEDELIJK                  8      0.97            1        3432.28            795.4
    10003  STEDELIJK                  2      1.53            2        7155.92           1254.6
    10004  VOORSTEDELIJK              9       2.4            0              0             1296
    10005  LANDELIJK                  5      0.99            0              0            356.4
    10006  VOORSTEDELIJK              1      0.83            0              0            448.2
    10007  LANDELIJK                  5      0.97            0              0            349.2
    10008  LANDELIJK                  7      2.32            0              0            835.2

... 92 more observatio


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.45 seconds
  cpu   0.45 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Stap 2 — Compileer de actuariële functiebibliotheek

Nu het hart van het notebook. `PROC FCMP` met `OUTLIB=work.actfuncs.pricing` compileert vier functies en één subroutine in het `pricing`-pakket van de dataset `work.actfuncs`:

- **`pure_premium`** — waargenomen schade per eenheid blootstelling (frequentie x ernst gecombineerd).
- **`credibility_z`** — credibiliteitsfactor volgens beperkte fluctuatie `Z = sqrt(n / (n + k))`, waarbij `n` de blootstellingsjaren van de polis is en `k` een afstemconstante.
- **`charged_premium`** — past een proportionele risico-opslag en een vaste kostenratio toe op een schadekost om tot de premie te komen die we daadwerkelijk in rekening brengen.
- **`pv_reserve`** — contante waarde van een toekomstige schade-uitkering, `FV / (1+r)**t`, gebruikt om schadereserves te verdisconteren.
- **`blend_premium`** (een SUBROUTINE) — gebruikt `OUTARGS` om *twee* waarden tegelijk terug te geven: de credibiliteitsgewogen zuivere premie en de gebruikte credibiliteitsfactor, zodat de DATA-stap beide in één aanroep krijgt.

Elke routine wordt afgesloten met `ENDSUB`, en de subroutine benoemt zijn schrijfbare argumenten met `OUTARGS`.

In [2]:
PROCEDURE fcmp outlib=work.actfuncs.pricing;

    /* Waargenomen zuivere premie: schadekost per eenheid blootstelling */
    function pure_premium(loss, exposure);
        ALS exposure <= 0 DAN RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Beperkte-fluctuatie credibiliteit: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        ALS n_years <= 0 DAN RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Gefactureerde premie = schadekost opgehoogd met risico-opslag + kosten */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        ALS expense_ratio >= 1 DAN RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Contante waarde van een toekomstige schade-uitkering, t jaar verdisconteerd tegen rente r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Credibiliteitsmenging: retourneert de gewogen zuivere premie EN de gebruikte Z */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

UITVOEREN;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Stap 3 — Registreer de bibliotheek met CMPLIB=

Het compileren van de routines is niet genoeg; SAS moet weten waar deze te vinden zijn wanneer een latere DATA-stap of procedure een naam gebruikt die niet als ingebouwd wordt herkend. De systeemoptie `CMPLIB=` verwijst naar de dataset (niet naar de driedelige pakketnaam) die de gecompileerde code bevat. Na deze `OPTIONS`-instructie is elke bovenstaande functie en subroutine aanroepbaar bij naam.

In [3]:
OPTIES cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Stap 4 — Tarifeer de portefeuille met de aangepaste functies

De tariferings-DATA-stap is nu bijna vrij van rekenwerk — de actuariële bedoeling is direct af te lezen aan de functienamen. Voor elke polis doen we het volgende:

1. Bereken de eigen waargenomen zuivere premie van de polis met `pure_premium`.
2. Roep de subroutine `blend_premium` aan om deze credibiliteitsgewogen af te wegen tegen het regionale klassetarief, waarbij zowel de gewogen schadekost als de gebruikte credibiliteitsfactor `z` via `OUTARGS` worden teruggegeven.
3. Verhoog de gewogen schadekost tot een gefactureerde premie met een risico-opslag van 12% en een kostenratio van 25% via `charged_premium`.
4. Schat de nog openstaande schadereserve als 35% van de geleden schade van de polis en verdisconteer deze drie jaar tegen 4% naar contante waarde met `pv_reserve`.

Merk op dat de uitvoerargumenten van de subroutine (`blended_pp`, `z`) moeten worden geïnitialiseerd vóór de `CALL`. De contante reservewaarde varieert per polis omdat deze wordt bepaald door de werkelijke geleden schade van elke polis — claimvrije polissen dragen een reserve van nul, dus hun `reserve_pv` is ook nul.

In [4]:
GEGEVENS rated;
    INSTELLEN portfolio;

    /* 1. Eigen schade-ervaring van de polis als zuivere premie */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Weeg eigen ervaring af tegen het klassetarief met credibiliteit.
          k = 4 blootstellingsjaren voor bijna volledige credibiliteit. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Zet de gewogen schadekost (per voertuigjaar) om naar gefactureerde premie */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Openstaande schadereserve = 35% van de geleden schade, verwacht
          af te wikkelen in 3 jaar; verdisconteer naar contante waarde tegen 4%. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    label
        own_pp       = "Eigen Zuivere Premie"
        blended_pp   = "Gewogen Zuivere Premie"
        z            = "Credibiliteitsfactor Z"
        premium      = "Premie"
        case_reserve = "Schadereserve"
        reserve_pv   = "Contante Waarde Reserve";

    BEWAREN policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
UITVOEREN;

PROCEDURE AFDRUKKEN GEGEVENS=rated(obs=10) noobs;
    TITEL "Getarifeerde Portefeuille (eerste 10 polissen)";
    VARIABELE policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
UITVOEREN;


                                     Getarifeerde Portefeuille (eerste 10 polissen)                                     

policy_id         region  years_insured  exposure   own_pp  blended_pp      z  premium  reserve_pv
    10001  LANDELIJK                  5      1.36        0       91.67  0.745   186.18           0
    10002  STEDELIJK                  8      0.97  3538.43     3039.59  0.816  4402.95     1067.95
    10003  STEDELIJK                  2      1.53  4677.07     3046.88  0.577  6961.51     2226.55
    10004  VOORSTEDELIJK              9       2.4        0       90.69  0.832   325.03           0
    10005  LANDELIJK                  5      0.99        0       91.67  0.745   135.52           0
    10006  VOORSTEDELIJK              1      0.83        0       298.5  0.447   369.98           0
    10007  LANDELIJK                  5      0.97        0       91.67  0.745   132.79           0
    10008  LANDELIJK                  7      2.32        0       72.82  0.798   252.29


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Stap 5 — Vat de getarifeerde portefeuille samen

Nu elke polis via dezelfde gecompileerde bibliotheek is getarifeerd, kunnen we de portefeuille per regio samenvatten. We rapporteren de gemiddelde gefactureerde premie, de gemiddelde credibiliteitsfactor, de totale geleden schade en de totale contant gemaakte schadereserve, en leiden vervolgens het impliciete schadepercentage per segment af, zodat we kunnen zien of de opgeslagen premie de verwachte schade dekt.

In [5]:
PROCEDURE GEMIDDELDEN GEGEVENS=rated mean sum maxdec=2 nonobs;
    KLASSE region;
    VARIABELE premium z incurred_loss reserve_pv;
    TITEL "Portefeuilleoverzicht per Tariferingsregio";
UITVOEREN;

/* Impliciete schadepercentage = geleden schade / gefactureerde premie, plus de
   contante reserve die het segment aanhoudt, per regio. */
PROCEDURE SQL;
    TITEL "Impliciete Schadepercentage en Contante Reservewaarde per Regio";
    SELECTEREN region label="Regio",
           count(*)                              AS n_policies label="Aantal Polissen",
           sum(incurred_loss)                    AS total_loss     OPMAAK=dollar12.2 label="Totale Schade",
           sum(premium)                          AS total_premium  OPMAAK=dollar12.2 label="Totale Premie",
           sum(incurred_loss) / sum(premium)     AS loss_ratio     OPMAAK=percent8.1 label="Schadepercentage",
           sum(reserve_pv)                       AS total_pv_reserve OPMAAK=dollar12.2 label="Totale Contante Reserve"
    FROM rated
    GROUP VOLGENS region
    ORDER VOLGENS region;
QUIT;
TITEL;


                                       Portefeuilleoverzicht per Tariferingsregio                                       

                                                  The MEANS Procedure

                                           Analysis Variable : premium Premie

        Regio                   Mean            Sum
        -------------------------------------------
        LANDELIJK             476.61       11438.62
        STEDELIJK            1987.17       67563.93
        VOORSTEDELIJK         813.04       34147.74
        -------------------------------------------

                                      Analysis Variable : z Credibiliteitsfactor Z

        Regio                   Mean            Sum
        -------------------------------------------
        LANDELIJK               0.71          17.14
        STEDELIJK               0.70          23.90
        VOORSTEDELIJK           0.68          28.36
        -------------------------------------------

                  


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## De resultaten interpreteren

De tariferings-DATA-stap schrijft nergens een enkele disconto- of credibiliteitsformule inline uit — hij roept alleen `pure_premium`, `blend_premium`, `charged_premium` en `pv_reserve` aan. Dat is de winst van **PROC FCMP**: de actuariële aannames leven in één gecompileerde bibliotheek die unit-getest, versiebeheerd en hergebruikt kan worden over tariferings-, reserverings- en rapportagewerk heen. Het wijzigen van de credibiliteitsconstante `k`, de risico-opslag of de kostenratio is een enkele wijziging in de bibliotheek, geen zoektocht door elk programma.

Bij het lezen van het getarifeerde bestand en het regionale overzicht:

- **Credibiliteit (`z`)** stijgt met `years_insured`, precies zoals de beperkte-fluctuatieformule `Z = sqrt(n / (n + k))` voorschrijft. Onder de eerste tien polissen behaalt de eenjarige voorstedelijke polis (10006) `z = 0,447`, de tweejarige stedelijke polis (10003) `z = 0,577`, terwijl de negenjarige voorstedelijke polis (10004) `z = 0,832` bereikt. Polissen met weinig ervaring worden naar het regionale klassetarief getrokken; langlopende polissen leunen op hun eigen schadeverleden.
- **Menging in de praktijk:** claimvrije polissen (het merendeel van het bestand) hebben `own_pp = 0`, dus `blend_premium` geeft een `blended_pp` terug die dicht bij `(1 - z)` maal het klassetarief ligt — bijv. polis 10001 (landelijk, `z = 0,745`) komt uit op `91,67` tegenover een klassetarief van `360`/voertuigjaar. De twee stedelijke polissen die wel schade hadden, 10002 en 10003, trekken hun gewogen schadekost juist omhoog richting hun eigen hoge ervaring (`3039,59` en `3046,88`).
- **Gefactureerde premie** ligt boven de gewogen schadekost omdat `charged_premium` een risico-opslag van 12% toevoegt en ophoogt voor een kostenratio van 25%, een vaste multiplier van `1,12 / 0,75 = 1,493` op de schadekost. De gemiddelde gefactureerde premie bedraagt `476,61` (landelijk), `813,04` (voorstedelijk) en `1.987,17` (stedelijk).
- **Verdisconteerde reserves:** `pv_reserve` verdisconteert de openstaande schadereserve van elke polis (35% van de geleden schade) drie jaar tegen 4%, een contante-waardefactor van `0,889`. Claimvrije polissen dragen `reserve_pv = 0`; de twee stedelijke schadegevallen dragen `1.067,95` en `2.226,55` bij. Opgeteld houdt het bestand een contant gemaakte reserve aan van `$2.151,56` (landelijk), `$5.932,52` (voorstedelijk) en `$13.364,13` (stedelijk).
- **Impliciete schadepercentages** komen uit op 60,5% (landelijk), 55,8% (voorstedelijk) en 63,6% (stedelijk) — allemaal ruim onder de 100%, dus de opgeslagen premie dekt de verwachte schade in elk segment. Het stedelijke segment loopt het heetst, gedreven door zijn twee grote gesimuleerde schadegevallen; een echte tariefherziening zou testen of dit signaal aanhoudt over meer schadejaren voordat het handmatige tarief wordt aangepast.

De subroutine `blend_premium` demonstreert ook het `OUTARGS`-idioom voor het teruggeven van meerdere resultaten uit één `CALL` — de gewogen schadekost en de credibiliteitsfactor `z` komen samen terug — waardoor aparte functieaanroepen worden vermeden en de tariferingslogica per polis compact en controleerbaar blijft.